# ML-Based Fault Classification for Waymo Crash Narratives

This notebook implements a fine-tuned transformer model (BERT/RoBERTa) for crash fault classification.

**Approach:**
1. Multi-task learning (predict fault + contact initiator + movement status)
2. Active learning loop for iterative improvement
3. Hybrid keyword + ML refinement layer

**Architecture:**
- Base: Pre-trained BERT/RoBERTa
- Task 1: Fault classification (4 classes)
- Task 2: Contact initiator (3 classes)
- Task 3: Movement status (3 classes)

## 1. Setup and Dependencies

In [ ]:
# Install required packages (uncomment if needed)
# !pip install transformers datasets torch scikit-learn pandas numpy matplotlib seaborn
# !pip install sentence-transformers  # for embeddings

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from transformers import (
    AutoTokenizer, 
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
from torch import nn
from torch.utils.data import DataLoader

import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Data Preparation

In [ ]:
# Load the crash data (adjust path as needed)
df = pd.read_csv('../data/waymo_crashes.csv')  # or load from your existing analysis

# Or load from existing analysis
# df = pd.read_pickle('../data/waymo_analysis.pkl')

print(f"Total crashes: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")

### 2.1 Create Label Schema

**Task 1: Fault Classification**
- 0: Waymo Primary Fault
- 1: Other Party Primary Fault
- 2: Shared Fault
- 3: Unclear/Insufficient Info

**Task 2: Contact Initiator**
- 0: Waymo Initiated
- 1: Other Party Initiated
- 2: Unclear

**Task 3: Movement Status**
- 0: Waymo Stationary
- 1: Waymo Moving
- 2: Unclear

In [ ]:
# Initialize label columns (all unlabeled at start)
df['fault_label'] = -1  # -1 means unlabeled
df['contact_initiator_label'] = -1
df['movement_status_label'] = -1

# Create a sample for initial labeling (stratified by existing keyword classification if available)
# Start with 200-300 examples for initial training
INITIAL_SAMPLE_SIZE = 250

# If you have keyword-based classification results, use them for stratification
if 'Contact Initiator' in df.columns:
    sample_df = df.groupby('Contact Initiator', group_keys=False).apply(
        lambda x: x.sample(min(len(x), INITIAL_SAMPLE_SIZE // 3), random_state=42)
    )
else:
    sample_df = df.sample(n=min(INITIAL_SAMPLE_SIZE, len(df)), random_state=42)

print(f"\nSample size for initial labeling: {len(sample_df)}")
print("\nNext step: Export this sample for manual labeling")

In [ ]:
# Export sample for manual labeling
sample_df[['Narrative', 'Crash With', 'fault_label', 'contact_initiator_label', 'movement_status_label']].to_csv(
    '../data/labeling_sample_batch1.csv', 
    index=False
)

print("Exported to: ../data/labeling_sample_batch1.csv")
print("\nInstructions:")
print("1. Open the CSV file")
print("2. Fill in the three label columns based on the narrative")
print("3. Save and load back into this notebook")
print("\nLabel values:")
print("  fault_label: 0=Waymo, 1=Other, 2=Shared, 3=Unclear")
print("  contact_initiator_label: 0=Waymo, 1=Other, 2=Unclear")
print("  movement_status_label: 0=Stationary, 1=Moving, 2=Unclear")

In [ ]:
# After manual labeling, load the labeled data
# labeled_df = pd.read_csv('../data/labeling_sample_batch1_LABELED.csv')

# For demonstration, we'll use keyword-based heuristics as proxy labels
# REPLACE THIS WITH YOUR MANUALLY LABELED DATA
def create_proxy_labels(df):
    """Create proxy labels from keyword classification for demonstration.
    Replace with your manually labeled data!"""
    
    df = df.copy()
    
    # Task 2: Contact Initiator (use existing keyword classification)
    if 'Contact Initiator' in df.columns:
        contact_map = {
            'Other party': 1,
            'Waymo': 0,
            'Unclear': 2
        }
        df['contact_initiator_label'] = df['Contact Initiator'].map(contact_map).fillna(2).astype(int)
    
    # Task 3: Movement Status
    if 'Waymo Movement' in df.columns:
        movement_map = {
            'Stationary': 0,
            'Proceeding': 1,
            'Unclear': 2
        }
        df['movement_status_label'] = df['Waymo Movement'].map(movement_map).fillna(2).astype(int)
    
    # Task 1: Fault (heuristic - MUST BE REPLACED WITH MANUAL LABELS)
    # This is a rough proxy - manual review is essential
    df['fault_label'] = 3  # Default to unclear
    
    # Simple heuristic: if other party initiated contact, likely their fault
    df.loc[(df['contact_initiator_label'] == 1) & (df['movement_status_label'] == 0), 'fault_label'] = 1
    # If Waymo initiated, might be Waymo's fault (but needs context)
    df.loc[(df['contact_initiator_label'] == 0), 'fault_label'] = 3  # Mark unclear for review
    
    return df

# Apply proxy labels (REPLACE WITH MANUAL LABELING)
labeled_df = create_proxy_labels(sample_df)

print(f"Labeled samples: {len(labeled_df)}")
print(f"\nFault label distribution:")
print(labeled_df['fault_label'].value_counts())
print(f"\nContact initiator distribution:")
print(labeled_df['contact_initiator_label'].value_counts())
print(f"\nMovement status distribution:")
print(labeled_df['movement_status_label'].value_counts())

## 3. Multi-Task Model Architecture

In [ ]:
class MultiTaskCrashClassifier(nn.Module):
    """Multi-task transformer model for crash fault classification.
    
    Tasks:
    1. Fault classification (4 classes)
    2. Contact initiator (3 classes)
    3. Movement status (3 classes)
    """
    
    def __init__(self, model_name='bert-base-uncased', dropout=0.3):
        super(MultiTaskCrashClassifier, self).__init__()
        
        # Load pre-trained transformer
        self.transformer = AutoModel.from_pretrained(model_name)
        hidden_size = self.transformer.config.hidden_size
        
        # Shared layers
        self.dropout = nn.Dropout(dropout)
        
        # Task-specific classification heads
        self.fault_classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 4)  # 4 classes: Waymo/Other/Shared/Unclear
        )
        
        self.contact_classifier = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 3)  # 3 classes: Waymo/Other/Unclear
        )
        
        self.movement_classifier = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 3)  # 3 classes: Stationary/Moving/Unclear
        )
    
    def forward(self, input_ids, attention_mask):
        # Get transformer outputs
        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # Use [CLS] token representation
        pooled_output = outputs.last_hidden_state[:, 0, :]  # [batch_size, hidden_size]
        pooled_output = self.dropout(pooled_output)
        
        # Task-specific predictions
        fault_logits = self.fault_classifier(pooled_output)
        contact_logits = self.contact_classifier(pooled_output)
        movement_logits = self.movement_classifier(pooled_output)
        
        return {
            'fault': fault_logits,
            'contact': contact_logits,
            'movement': movement_logits
        }

print("Multi-task model architecture defined.")

## 4. Data Processing and Tokenization

In [ ]:
# Initialize tokenizer
MODEL_NAME = 'bert-base-uncased'  # or 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples['Narrative'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

# Prepare dataset
def prepare_dataset(df):
    """Convert pandas DataFrame to HuggingFace Dataset."""
    
    # Filter out unlabeled examples
    df_clean = df[
        (df['fault_label'] >= 0) & 
        (df['contact_initiator_label'] >= 0) & 
        (df['movement_status_label'] >= 0)
    ].copy()
    
    # Create dataset
    dataset = Dataset.from_pandas(df_clean[[
        'Narrative', 
        'fault_label', 
        'contact_initiator_label', 
        'movement_status_label'
    ]])
    
    # Tokenize
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    
    # Rename labels for easier access
    tokenized_dataset = tokenized_dataset.rename_column('fault_label', 'labels_fault')
    tokenized_dataset = tokenized_dataset.rename_column('contact_initiator_label', 'labels_contact')
    tokenized_dataset = tokenized_dataset.rename_column('movement_status_label', 'labels_movement')
    
    return tokenized_dataset

# Prepare and split data
dataset = prepare_dataset(labeled_df)

# Split into train/val/test (70/15/15)
train_test = dataset.train_test_split(test_size=0.3, seed=42)
test_val = train_test['test'].train_test_split(test_size=0.5, seed=42)

train_dataset = train_test['train']
val_dataset = test_val['train']
test_dataset = test_val['test']

print(f"Train set: {len(train_dataset)}")
print(f"Validation set: {len(val_dataset)}")
print(f"Test set: {len(test_dataset)}")

## 5. Training Loop with Multi-Task Loss

In [ ]:
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from tqdm import tqdm

class MultiTaskTrainer:
    """Custom trainer for multi-task learning."""
    
    def __init__(self, model, train_dataset, val_dataset, device, 
                 batch_size=16, learning_rate=2e-5, task_weights=None):
        self.model = model.to(device)
        self.device = device
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        
        # Task weights (adjust based on importance or class imbalance)
        if task_weights is None:
            self.task_weights = {
                'fault': 1.0,      # Primary task
                'contact': 0.5,    # Auxiliary task
                'movement': 0.3    # Auxiliary task
            }
        else:
            self.task_weights = task_weights
        
        # Loss functions
        self.fault_criterion = CrossEntropyLoss()
        self.contact_criterion = CrossEntropyLoss()
        self.movement_criterion = CrossEntropyLoss()
        
        # Optimizer
        self.optimizer = AdamW(model.parameters(), lr=learning_rate)
        
        # Data loaders
        self.train_loader = DataLoader(
            train_dataset, 
            batch_size=batch_size, 
            shuffle=True
        )
        self.val_loader = DataLoader(
            val_dataset, 
            batch_size=batch_size
        )
    
    def compute_loss(self, outputs, batch):
        """Compute weighted multi-task loss."""
        
        # Individual task losses
        loss_fault = self.fault_criterion(
            outputs['fault'], 
            batch['labels_fault'].to(self.device)
        )
        loss_contact = self.contact_criterion(
            outputs['contact'], 
            batch['labels_contact'].to(self.device)
        )
        loss_movement = self.movement_criterion(
            outputs['movement'], 
            batch['labels_movement'].to(self.device)
        )
        
        # Weighted combination
        total_loss = (
            self.task_weights['fault'] * loss_fault +
            self.task_weights['contact'] * loss_contact +
            self.task_weights['movement'] * loss_movement
        )
        
        return total_loss, {
            'fault': loss_fault.item(),
            'contact': loss_contact.item(),
            'movement': loss_movement.item(),
            'total': total_loss.item()
        }
    
    def train_epoch(self):
        """Train for one epoch."""
        self.model.train()
        total_loss = 0
        
        for batch in tqdm(self.train_loader, desc="Training"):
            # Forward pass
            outputs = self.model(
                input_ids=batch['input_ids'].to(self.device),
                attention_mask=batch['attention_mask'].to(self.device)
            )
            
            # Compute loss
            loss, _ = self.compute_loss(outputs, batch)
            
            # Backward pass
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            
            total_loss += loss.item()
        
        return total_loss / len(self.train_loader)
    
    def evaluate(self):
        """Evaluate on validation set."""
        self.model.eval()
        total_loss = 0
        
        predictions = {'fault': [], 'contact': [], 'movement': []}
        labels = {'fault': [], 'contact': [], 'movement': []}
        
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Evaluating"):
                outputs = self.model(
                    input_ids=batch['input_ids'].to(self.device),
                    attention_mask=batch['attention_mask'].to(self.device)
                )
                
                loss, _ = self.compute_loss(outputs, batch)
                total_loss += loss.item()
                
                # Store predictions
                predictions['fault'].extend(outputs['fault'].argmax(dim=1).cpu().numpy())
                predictions['contact'].extend(outputs['contact'].argmax(dim=1).cpu().numpy())
                predictions['movement'].extend(outputs['movement'].argmax(dim=1).cpu().numpy())
                
                # Store labels
                labels['fault'].extend(batch['labels_fault'].numpy())
                labels['contact'].extend(batch['labels_contact'].numpy())
                labels['movement'].extend(batch['labels_movement'].numpy())
        
        avg_loss = total_loss / len(self.val_loader)
        
        # Compute accuracies
        accuracies = {
            'fault': accuracy_score(labels['fault'], predictions['fault']),
            'contact': accuracy_score(labels['contact'], predictions['contact']),
            'movement': accuracy_score(labels['movement'], predictions['movement'])
        }
        
        return avg_loss, accuracies, predictions, labels
    
    def train(self, num_epochs=5, patience=3):
        """Full training loop with early stopping."""
        best_val_loss = float('inf')
        patience_counter = 0
        
        history = {
            'train_loss': [],
            'val_loss': [],
            'val_acc_fault': [],
            'val_acc_contact': [],
            'val_acc_movement': []
        }
        
        for epoch in range(num_epochs):
            print(f"\n{'='*60}")
            print(f"Epoch {epoch + 1}/{num_epochs}")
            print(f"{'='*60}")
            
            # Train
            train_loss = self.train_epoch()
            
            # Evaluate
            val_loss, accuracies, _, _ = self.evaluate()
            
            # Store history
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['val_acc_fault'].append(accuracies['fault'])
            history['val_acc_contact'].append(accuracies['contact'])
            history['val_acc_movement'].append(accuracies['movement'])
            
            # Print metrics
            print(f"\nTrain Loss: {train_loss:.4f}")
            print(f"Val Loss: {val_loss:.4f}")
            print(f"Val Accuracy - Fault: {accuracies['fault']:.4f}")
            print(f"Val Accuracy - Contact: {accuracies['contact']:.4f}")
            print(f"Val Accuracy - Movement: {accuracies['movement']:.4f}")
            
            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                # Save best model
                torch.save(self.model.state_dict(), '../models/best_model.pt')
                print("✓ Saved best model")
            else:
                patience_counter += 1
                print(f"Patience: {patience_counter}/{patience}")
                
                if patience_counter >= patience:
                    print(f"\nEarly stopping triggered at epoch {epoch + 1}")
                    break
        
        return history

print("Multi-task trainer defined.")

## 6. Train the Model

In [ ]:
# Initialize model
model = MultiTaskCrashClassifier(model_name=MODEL_NAME)

# Initialize trainer
trainer = MultiTaskTrainer(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    device=device,
    batch_size=16,
    learning_rate=2e-5
)

# Train
history = trainer.train(num_epochs=10, patience=3)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracies
axes[1].plot(history['val_acc_fault'], label='Fault', marker='o')
axes[1].plot(history['val_acc_contact'], label='Contact', marker='s')
axes[1].plot(history['val_acc_movement'], label='Movement', marker='^')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy by Task')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('../figures/training_history.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Evaluation on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('../models/best_model.pt'))
model.eval()

# Create test loader
test_loader = DataLoader(test_dataset, batch_size=16)

# Evaluate
test_predictions = {'fault': [], 'contact': [], 'movement': []}
test_labels = {'fault': [], 'contact': [], 'movement': []}

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        outputs = model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device)
        )
        
        test_predictions['fault'].extend(outputs['fault'].argmax(dim=1).cpu().numpy())
        test_predictions['contact'].extend(outputs['contact'].argmax(dim=1).cpu().numpy())
        test_predictions['movement'].extend(outputs['movement'].argmax(dim=1).cpu().numpy())
        
        test_labels['fault'].extend(batch['labels_fault'].numpy())
        test_labels['contact'].extend(batch['labels_contact'].numpy())
        test_labels['movement'].extend(batch['labels_movement'].numpy())

# Print classification reports
task_names = {
    'fault': ['Waymo Fault', 'Other Fault', 'Shared', 'Unclear'],
    'contact': ['Waymo Initiated', 'Other Initiated', 'Unclear'],
    'movement': ['Stationary', 'Moving', 'Unclear']
}

for task in ['fault', 'contact', 'movement']:
    print(f"\n{'='*60}")
    print(f"Task: {task.upper()}")
    print(f"{'='*60}")
    print(classification_report(
        test_labels[task], 
        test_predictions[task],
        target_names=task_names[task],
        digits=3
    ))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (task, ax) in enumerate(zip(['fault', 'contact', 'movement'], axes)):
    cm = confusion_matrix(test_labels[task], test_predictions[task])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=task_names[task],
                yticklabels=task_names[task])
    ax.set_title(f'{task.capitalize()} Classification')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('../figures/confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Active Learning Loop

In [ ]:
class ActiveLearner:
    """Active learning for iterative model improvement.
    
    Strategies:
    1. Uncertainty sampling (low confidence)
    2. Disagreement sampling (predictions disagree across tasks)
    3. Representative sampling (diverse examples)
    """
    
    def __init__(self, model, tokenizer, device, unlabeled_df):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.unlabeled_df = unlabeled_df.copy()
    
    def predict_with_confidence(self, narratives):
        """Get predictions with confidence scores."""
        self.model.eval()
        
        predictions = []
        confidences = []
        
        with torch.no_grad():
            for narrative in tqdm(narratives, desc="Predicting"):
                # Tokenize
                inputs = self.tokenizer(
                    narrative,
                    return_tensors='pt',
                    padding='max_length',
                    truncation=True,
                    max_length=512
                ).to(self.device)
                
                # Predict
                outputs = self.model(**inputs)
                
                # Get probabilities
                probs = {
                    'fault': torch.softmax(outputs['fault'], dim=1)[0],
                    'contact': torch.softmax(outputs['contact'], dim=1)[0],
                    'movement': torch.softmax(outputs['movement'], dim=1)[0]
                }
                
                # Get predictions and confidence
                pred = {
                    'fault': probs['fault'].argmax().item(),
                    'contact': probs['contact'].argmax().item(),
                    'movement': probs['movement'].argmax().item()
                }
                
                conf = {
                    'fault': probs['fault'].max().item(),
                    'contact': probs['contact'].max().item(),
                    'movement': probs['movement'].max().item()
                }
                
                predictions.append(pred)
                confidences.append(conf)
        
        return predictions, confidences
    
    def uncertainty_sampling(self, n_samples=50):
        """Select examples with lowest confidence (most uncertain)."""
        
        predictions, confidences = self.predict_with_confidence(
            self.unlabeled_df['Narrative'].tolist()
        )
        
        # Calculate average confidence across tasks
        avg_confidences = [
            np.mean([c['fault'], c['contact'], c['movement']])
            for c in confidences
        ]
        
        # Get indices of lowest confidence
        uncertain_indices = np.argsort(avg_confidences)[:n_samples]
        
        return self.unlabeled_df.iloc[uncertain_indices]
    
    def disagreement_sampling(self, n_samples=50):
        """Select examples where task predictions might disagree."""
        
        predictions, confidences = self.predict_with_confidence(
            self.unlabeled_df['Narrative'].tolist()
        )
        
        # Calculate disagreement score
        disagreement_scores = []
        for pred in predictions:
            # Check if predictions suggest different interpretations
            # Example: contact says "Other initiated" but fault says "Waymo"
            score = 0
            
            # High disagreement if contact=Other but fault=Waymo
            if pred['contact'] == 1 and pred['fault'] == 0:
                score += 2
            
            # Or if stationary but Waymo initiated
            if pred['movement'] == 0 and pred['contact'] == 0:
                score += 1
            
            disagreement_scores.append(score)
        
        # Get highest disagreement
        disagreement_indices = np.argsort(disagreement_scores)[-n_samples:]
        
        return self.unlabeled_df.iloc[disagreement_indices]
    
    def get_next_batch(self, n_samples=50, strategy='uncertainty'):
        """Get next batch of examples to label."""
        
        if strategy == 'uncertainty':
            return self.uncertainty_sampling(n_samples)
        elif strategy == 'disagreement':
            return self.disagreement_sampling(n_samples)
        else:
            # Random sampling as baseline
            return self.unlabeled_df.sample(n=n_samples)

print("Active learning module defined.")

In [ ]:
# Get unlabeled data
unlabeled_df = df[df['fault_label'] == -1].copy()

print(f"Unlabeled examples: {len(unlabeled_df)}")

# Initialize active learner
active_learner = ActiveLearner(
    model=model,
    tokenizer=tokenizer,
    device=device,
    unlabeled_df=unlabeled_df
)

# Get next batch for labeling
next_batch = active_learner.get_next_batch(n_samples=50, strategy='uncertainty')

# Export for labeling
next_batch[['Narrative', 'Crash With', 'fault_label', 'contact_initiator_label', 'movement_status_label']].to_csv(
    '../data/labeling_sample_batch2.csv',
    index=False
)

print("\nExported next batch to: ../data/labeling_sample_batch2.csv")
print("After labeling, retrain the model with the combined dataset.")

## 9. Hybrid Approach: Keyword + ML Refinement

In [ ]:
class HybridClassifier:
    """Hybrid keyword + ML classifier.
    
    Strategy:
    1. Use keyword patterns for clear cases (high confidence)
    2. Use ML model for unclear cases
    3. Use ML to validate/refine keyword predictions
    """
    
    def __init__(self, ml_model, tokenizer, device):
        self.ml_model = ml_model
        self.tokenizer = tokenizer
        self.device = device
    
    def keyword_classify(self, narrative):
        """Apply keyword-based classification."""
        narrative_lower = narrative.lower()
        
        # Contact initiator patterns
        other_initiated_patterns = [
            'made contact with the rear of the waymo',
            'made contact with the front of the waymo',
            'made contact with the stationary waymo',
            'struck the waymo',
            'struck the rear of the waymo'
        ]
        
        waymo_initiated_patterns = [
            'waymo av made contact with',
            'waymo av struck',
            'of the waymo made contact with'
        ]
        
        # Check patterns
        contact_initiator = 'unclear'
        confidence = 0.0
        
        for pattern in other_initiated_patterns:
            if pattern in narrative_lower:
                contact_initiator = 'other'
                confidence = 0.9  # High confidence for keyword match
                break
        
        if contact_initiator == 'unclear':
            for pattern in waymo_initiated_patterns:
                if pattern in narrative_lower:
                    contact_initiator = 'waymo'
                    confidence = 0.8
                    break
        
        return {
            'contact_initiator': contact_initiator,
            'confidence': confidence
        }
    
    def ml_classify(self, narrative):
        """Apply ML-based classification."""
        self.ml_model.eval()
        
        with torch.no_grad():
            # Tokenize
            inputs = self.tokenizer(
                narrative,
                return_tensors='pt',
                padding='max_length',
                truncation=True,
                max_length=512
            ).to(self.device)
            
            # Predict
            outputs = self.ml_model(**inputs)
            
            # Get probabilities
            probs = {
                'fault': torch.softmax(outputs['fault'], dim=1)[0],
                'contact': torch.softmax(outputs['contact'], dim=1)[0],
                'movement': torch.softmax(outputs['movement'], dim=1)[0]
            }
            
            return {
                'fault': {
                    'prediction': probs['fault'].argmax().item(),
                    'confidence': probs['fault'].max().item()
                },
                'contact': {
                    'prediction': probs['contact'].argmax().item(),
                    'confidence': probs['contact'].max().item()
                },
                'movement': {
                    'prediction': probs['movement'].argmax().item(),
                    'confidence': probs['movement'].max().item()
                }
            }
    
    def classify(self, narrative, confidence_threshold=0.7):
        """Hybrid classification with decision logic."""
        
        # Step 1: Try keyword classification
        keyword_result = self.keyword_classify(narrative)
        
        # Step 2: Get ML prediction
        ml_result = self.ml_classify(narrative)
        
        # Step 3: Decide which to use
        final_result = {
            'method': '',
            'fault': None,
            'contact': None,
            'movement': None,
            'needs_review': False
        }
        
        # For contact initiator: use keyword if available, otherwise ML
        if keyword_result['confidence'] > confidence_threshold:
            final_result['contact'] = keyword_result['contact_initiator']
            final_result['method'] = 'keyword'
        else:
            final_result['contact'] = ml_result['contact']['prediction']
            final_result['method'] = 'ml'
            
            # Flag for review if ML confidence is also low
            if ml_result['contact']['confidence'] < confidence_threshold:
                final_result['needs_review'] = True
        
        # For fault and movement: always use ML
        final_result['fault'] = ml_result['fault']['prediction']
        final_result['movement'] = ml_result['movement']['prediction']
        
        # Cross-validation: flag disagreement between keyword and ML
        if keyword_result['contact_initiator'] != 'unclear':
            keyword_to_idx = {'waymo': 0, 'other': 1, 'unclear': 2}
            if keyword_to_idx[keyword_result['contact_initiator']] != ml_result['contact']['prediction']:
                final_result['needs_review'] = True
                final_result['disagreement'] = {
                    'keyword': keyword_result['contact_initiator'],
                    'ml': ml_result['contact']['prediction']
                }
        
        return final_result

print("Hybrid classifier defined.")

In [ ]:
# Initialize hybrid classifier
hybrid_classifier = HybridClassifier(
    ml_model=model,
    tokenizer=tokenizer,
    device=device
)

# Test on a few examples
test_narratives = df.sample(5)['Narrative'].tolist()

for i, narrative in enumerate(test_narratives):
    print(f"\n{'='*80}")
    print(f"Example {i+1}")
    print(f"{'='*80}")
    print(f"Narrative: {narrative[:200]}...")
    
    result = hybrid_classifier.classify(narrative)
    
    print(f"\nResult:")
    print(f"  Method: {result['method']}")
    print(f"  Contact: {result['contact']}")
    print(f"  Fault: {result['fault']}")
    print(f"  Movement: {result['movement']}")
    print(f"  Needs Review: {result['needs_review']}")
    if 'disagreement' in result:
        print(f"  ⚠️  Disagreement: {result['disagreement']}")

## 10. Apply to Full Dataset

In [ ]:
# Apply hybrid classifier to all crashes
results = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying all crashes"):
    result = hybrid_classifier.classify(row['Narrative'])
    results.append(result)

# Add results to dataframe
df['ml_fault'] = [r['fault'] for r in results]
df['ml_contact'] = [r['contact'] for r in results]
df['ml_movement'] = [r['movement'] for r in results]
df['ml_needs_review'] = [r['needs_review'] for r in results]
df['ml_method'] = [r['method'] for r in results]

print("\nML Classification Results:")
print(f"\nFault distribution:")
print(df['ml_fault'].value_counts())
print(f"\nContact initiator distribution:")
print(df['ml_contact'].value_counts())
print(f"\nMovement distribution:")
print(df['ml_movement'].value_counts())
print(f"\nCases needing review: {df['ml_needs_review'].sum()} ({df['ml_needs_review'].sum()/len(df)*100:.1f}%)")

In [ ]:
# Save results
df.to_csv('../data/waymo_crashes_ml_classified.csv', index=False)
print("Results saved to: ../data/waymo_crashes_ml_classified.csv")

## 11. Analysis and Comparison

In [ ]:
# Compare keyword-based vs ML-based classification (if keyword results exist)
if 'Contact Initiator' in df.columns:
    # Map to numeric
    keyword_map = {'Other party': 1, 'Waymo': 0, 'Unclear': 2}
    df['keyword_contact'] = df['Contact Initiator'].map(keyword_map)
    
    # Compare
    agreement = (df['keyword_contact'] == df['ml_contact']).sum()
    print(f"\nKeyword vs ML Agreement: {agreement}/{len(df)} ({agreement/len(df)*100:.1f}%)")
    
    # Show disagreements
    disagreements = df[df['keyword_contact'] != df['ml_contact']]
    print(f"\nDisagreements: {len(disagreements)}")
    
    if len(disagreements) > 0:
        print("\nSample disagreements:")
        for idx, row in disagreements.head(3).iterrows():
            print(f"\n{'-'*80}")
            print(f"Narrative: {row['Narrative'][:200]}...")
            print(f"Keyword: {row['Contact Initiator']}")
            print(f"ML: {row['ml_contact']}")

## 12. Next Steps

**Immediate:**
1. Manually label 200-300 narratives with ground truth fault classifications
2. Retrain model with real labels (not proxy labels)
3. Evaluate accuracy on held-out test set

**Active Learning Iterations:**
1. Run active learning to select uncertain cases (Section 8)
2. Label selected cases
3. Add to training set and retrain
4. Repeat until performance plateaus

**Model Improvements:**
1. Try different base models (RoBERTa, ALBERT, domain-adapted BERT)
2. Experiment with task weights in multi-task learning
3. Add attention visualization to understand what model focuses on
4. Implement ensemble methods (multiple models voting)

**Deployment:**
1. Export model for production use
2. Build API endpoint for real-time classification
3. Create confidence-based routing (high confidence → auto-classify, low → human review)
4. Monitor and collect feedback for continuous improvement

In [ ]:
# Export model for deployment
import os
os.makedirs('../models', exist_ok=True)

# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'model_name': MODEL_NAME,
}, '../models/crash_classifier_final.pt')

# Save tokenizer
tokenizer.save_pretrained('../models/tokenizer')

print("Model and tokenizer saved for deployment.")